# Avaliação Industrial com IA — Google Colab

Notebook automatizado com barras de progresso e logs em tempo real para clone, instalação, análise, treinamento tabular, testes, download/preparação visual e épocas YOLO. **Recomendado: T4 GPU** para o treinamento YOLO opcional. A etapa tabular usa CPU.

No Colab, selecione **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU** antes de executar. A etapa visual é opcional porque baixa um dataset e pode levar bastante tempo.

In [ ]:
# Configuração da execução
RUN_VISION = False  # True para baixar/preparar dataset e treinar YOLO
VISION_EPOCHS = 50
VISION_IMGSZ = 640
VISION_BATCH = 16  # reduza para 8 se ocorrer falta de memória na T4
REPO_URL = 'https://github.com/Augusto-Dantas/avaliacao-industrial-ia.git'
REPO_DIR = '/content/avaliacao-industrial-ia'


In [ ]:
import os, sys, subprocess, pathlib, re, time
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm>=4.66'], check=True)
from tqdm.auto import tqdm

def run_stream(command, title, total=None, epoch_mode=False, env=None):
    """Executa um comando mostrando logs e progresso reconhecido na saída."""
    bar = tqdm(total=total, desc=title, unit="etapa" if not epoch_mode else "época",
               dynamic_ncols=True) if total else None
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, cwd=os.getcwd(), env=env
    )
    last = 0
    try:
        for raw in process.stdout:
            line = raw.rstrip()
            if line:
                tqdm.write(line)
            if epoch_mode:
                match = re.match(r"^\s*(\d+)\s*/\s*(\d+)\s+", line)
                if match:
                    current, maximum = map(int, match.groups())
                    if bar is None:
                        bar = tqdm(total=maximum, desc=title, unit="época", dynamic_ncols=True)
                    if bar.total != maximum:
                        bar.total = maximum
                    if current > last:
                        bar.update(current - last)
                        last = current
            elif bar is not None and total:
                if total == 100:
                    match = re.search(r"(\d{1,3})%", line)
                    if match:
                        current = min(100, int(match.group(1)))
                        if current > last:
                            bar.update(current - last)
                            last = current
                else:
                    match = re.search(r"\[(\d+)/(\d+)\]", line)
                    if match:
                        current, maximum = map(int, match.groups())
                        if maximum == total and current > last:
                            bar.update(current - last)
                            last = current
        code = process.wait()
        if code != 0:
            raise subprocess.CalledProcessError(code, command)
        if bar is not None and not epoch_mode and bar.n < bar.total:
            bar.update(bar.total - bar.n)
    finally:
        if bar is not None:
            bar.close()

print('Python:', sys.version)
if os.path.isdir(REPO_DIR):
    run_stream(['git', '-C', REPO_DIR, 'pull', '--ff-only'], 'Atualizando repositório', total=1)
else:
    run_stream(['git', 'clone', '--progress', '--depth', '1', REPO_URL, REPO_DIR],
               'Clonando repositório', total=100)
%cd /content/avaliacao-industrial-ia

In [ ]:
# O Colab gerencia o Python do runtime: não criar virtualenv/ensurepip.
run_stream([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-dev.txt'],
           'Instalando dependências', total=1)
if RUN_VISION:
    run_stream([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-vision.txt'],
               'Instalando dependências de visão', total=1)

In [ ]:
# O script informa [1/5] ... [5/5]; a barra acompanha essas etapas reais.
env = os.environ.copy()
env['USE_SYSTEM_PYTHON'] = '1'
env['PYTHON_BIN'] = sys.executable
run_stream(['bash', 'scripts/run_pipeline.sh', '--no-install'],
           'Pipeline tabular', total=5, env=env)

In [ ]:
if RUN_VISION:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA indisponível. Selecione T4 GPU no Colab e reinicie o runtime.')
    print('GPU:', torch.cuda.get_device_name(0))
    # Primeiro baixa/prepara o dataset. O treinamento será executado neste kernel
    # para que as barras tqdm sejam renderizadas dinamicamente pelo próprio Colab.
    prepare_cmd = [
        sys.executable, 'src/setup_yolo.py', '--no-install',
        '--datasets', 'rikkarth/welding-defect-object-detection'
    ]
    run_stream(prepare_cmd, 'Preparando dataset YOLO')

    from ultralytics import YOLO
    from tqdm.auto import tqdm
    model = YOLO('yolo26n.pt')
    epoch_bar = tqdm(total=VISION_EPOCHS, desc='Épocas YOLO', unit='época', dynamic_ncols=True)
    batch_bar = {'bar': None, 'last_epoch': -1}

    def on_epoch_start(trainer):
        if batch_bar['bar'] is not None:
            batch_bar['bar'].close()
        batch_bar['last_epoch'] = trainer.epoch
        batch_bar['bar'] = tqdm(total=trainer.nb, desc=f'Época {trainer.epoch + 1}/{trainer.epochs}',
                                unit='batch', leave=False, dynamic_ncols=True)

    def on_batch_end(trainer):
        bar = batch_bar['bar']
        if bar is not None:
            bar.update(1)
            try:
                losses = trainer.tloss.detach().float().cpu().flatten().tolist()
                bar.set_postfix(loss=f'{sum(losses):.4f}')
            except Exception:
                pass

    def on_epoch_end(trainer):
        epoch_bar.update(1)
        try:
            metrics = {k: round(float(v), 4) for k, v in trainer.metrics.items()
                       if isinstance(v, (int, float)) or hasattr(v, '__float__')}
            if metrics:
                epoch_bar.set_postfix(metrics)
        except Exception:
            pass

    model.add_callback('on_train_epoch_start', on_epoch_start)
    model.add_callback('on_train_batch_end', on_batch_end)
    model.add_callback('on_train_epoch_end', on_epoch_end)
    try:
        model.train(
            data='data/yolo/data.yaml', epochs=VISION_EPOCHS, imgsz=VISION_IMGSZ,
            batch=VISION_BATCH, device=0, workers=4, project='runs/detect',
            name='welding-yolo', exist_ok=True, plots=True, seed=42,
            save=True, save_period=1, verbose=False
        )
    finally:
        if batch_bar['bar'] is not None:
            batch_bar['bar'].close()
        epoch_bar.close()
else:
    print('Treinamento YOLO ignorado (RUN_VISION=False).')

In [ ]:
from pathlib import Path
import zipfile
artifact_paths = [Path('reports'), Path('models'), Path('runs/detect/welding-yolo')]
files_to_pack = [
    file for base in artifact_paths if base.exists()
    for file in base.rglob('*')
    if file.is_file() and '.cache' not in file.parts
]
archive = Path('/content/avaliacao-industrial-ia-artifacts.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for file in tqdm(files_to_pack, desc='Compactando artefatos', unit='arquivo'):
        zf.write(file, file.as_posix())
print(f'Artefatos exportados: {archive} ({archive.stat().st_size / 1024**2:.1f} MiB)')
from google.colab import files
files.download(str(archive))

## Interpretação e limites

- Os modelos tabulares são treinados no CSV industrial versionado.
- O detector visual usa dataset auxiliar com classes genéricas (bad_weld, good_weld, defect). Não alegue que ele identifica especificamente porosidade nem mede HAZ térmica.
- O runtime Colab é temporário. Baixe o ZIP ao final; arquivos em /content podem desaparecer quando a sessão terminar.
